In [1]:
'''
Система оценки стоимости объектов недвижимости 
на основе алгоритма градивентного бустинга (ensemble)
'''

# импорт библиотек
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import ensemble
from sklearn.metrics import mean_absolute_error

# импорт набора данных
df = pd.read_csv('T:\Machine Learning\Книга\Data Base\Melbourne_housing_FULL\Melbourne_housing_FULL.csv')

# очистка набора данных
del df['Address']
del df['Method']
del df['SellerG']
del df['Date']
del df['Postcode']
del df['Lattitude']
del df['Longtitude']
del df['Regionname']
del df['Propertycount']

# удаление строк с пропущенными значениями

# df.dropna(axis = 0, how='any', thresh=None, 
#           subset=None, inplace=True)

df.dropna(axis=0, how='any', inplace=True)

'''
axis: 0 - удаляет строки с пропущенными значениями
      1 - удаляет столбцы с пропущенными значениями

how: any - удаляет строки или столбцы с любым количеством пропущенныз значений
     all - удаляет строки или столбцы, все знаяения которых пропущены

thresh: integer - задает целочисленное пороговое знаяение для удаления
                    столбцов/строк
        None - не задает пороговое значение

subset: variable - определяет, в каких столбцах следует искать отсутствующее значение
        None - не указывает подмножество

inplace: True - операция выполняется на месте 
                    (выполняется обновление, а не замена)
         False
'''

# преобразование столбцов, содержащих не числовые данные, в числовые значения
df = pd.get_dummies(df, columns=['Suburb', 'CouncilArea', 'Type'])

# назначение зависимых и независимых переменных
X = df.drop('Price', axis=1)
y = df['Price']

# разбиение набора данных (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, 
                                                    shuffle=True)

# выбор алгоритма и настройка его гиперпараметров
model = ensemble.GradientBoostingRegressor(
    n_estimators=150,       # задает количество деревьев решений
    learning_rate=0.1,      # определяет степень влияния дополнительных деревьев на общий прогноз
    max_depth=30,           # определяте максимальное количество уровней (глубину) каждого дерева решений
    min_samples_split=4,    # определяет минимальное количество образцов, необходимое для выполнения очередного бинарного разделения
    min_samples_leaf=6,     # определяет минимальное количество образцов, которые должны появиться в каждом дочернем узле (листе), прежде чем будет создана новая ветвь
    max_features=0.6,       # определяет общее количество признаков, предъявляемых модели при определении наилучшего разбиения
    loss='huber'            # вычисляет коэффициент ошибок модели (huber, squared_error, absolute_error, quantile)
)

# связь тренировочных данных с алгоритмом обучения
model.fit(X_train, y_train)

# оценка результатов
mae_train = mean_absolute_error(y_train, 
                                model.predict(X_train))
print("Training Set Mean Absolute Error: %.2f" %
      mae_train)

mae_test = mean_absolute_error(y_test,
                               model.predict(X_test))
print("Test Set Mean Absolute Error: %.2f" %
      mae_test)

Training Set Mean Absolute Error: 27827.98
Test Set Mean Absolute Error: 170795.25


In [2]:
'''
Оптимизация модели 
для уменьшения последствий переобучения
'''

model = ensemble.GradientBoostingRegressor(
    n_estimators=250,       
    learning_rate=0.1,      
    max_depth=5,           
    min_samples_split=4,    
    min_samples_leaf=6,     
    max_features=0.6,       
    loss='huber'            
)

model.fit(X_train, y_train)

mae_train = mean_absolute_error(y_train, 
                                model.predict(X_train))
print("Training Set Mean Absolute Error: %.2f" %
      mae_train)

mae_test = mean_absolute_error(y_test,
                               model.predict(X_test))
print("Test Set Mean Absolute Error: %.2f" %
      mae_test)

Training Set Mean Absolute Error: 122166.02
Test Set Mean Absolute Error: 161480.40


In [ ]:
'''
Код для выполнения поиска по решетке
'''

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import ensemble
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV

df = pd.read_csv('T:\Machine Learning\Книга\Data Base\Melbourne_housing_FULL\Melbourne_housing_FULL.csv')

del df['Address']
del df['Method']
del df['SellerG']
del df['Date']
del df['Postcode']
del df['Lattitude']
del df['Longtitude']
del df['Regionname']
del df['Propertycount']

df.dropna(axis=0, how='any', inplace=True)

df = pd.get_dummies(df, columns=['Suburb', 'CouncilArea', 'Type'])

X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, 
                                                    shuffle=True)

model = ensemble.GradientBoostingRegressor()
hyperparameters = {
    'n_estimators': [200, 300],
    'max_depth': [4, 6],
    'min_samples_split': [3, 4],
    'min_samples_leaf': [5, 6],
    'learning_rate': [0.01, 0.02],
    'max_features': [0.8, 0.9],
    'loss': ['squared_error', 'absolute_error', 'huber']
}

grid = GridSearchCV(model, hyperparameters, n_jobs=4)

grid.fit(X_train, y_train)

grid.best_params_

mae_train = mean_absolute_error(y_train, 
                                grid.predict(X_train))
print("Training Set Mean Absolute Error: %.2f" %
      mae_train)

mae_test = mean_absolute_error(y_test,
                               grid.predict(X_test))
print("Test Set Mean Absolute Error: %.2f" %
      mae_test)